# Sub 04: Model 1 + Partial Layer-Wise Fine-Tuning
### Partial fine-tuning: layers 1 to 8 frozen; top layers 9 to 16 and classification head unfrozen.

* **Course:** CS781: Deep Learning for Natural Language Processing (IIT Kanpur)
* **Instructor:** Prof. Ashutosh Modi
* **Target Objective:** Strict Macro-F1 Leaderboard Champion


In [ ]:
"""
CS781 Capstone Phase 2 Legal Named Entity Recognition (L-NER)
Sub 04: Model 1 + Partial Layer-Wise Fine-Tuning
Target Metric: Strict Macro-F1 Leaderboard Champion
Architecture: LLAMA (model1-deep-llama-135m)
Method: PARTIAL_FT | Loss: CE
"""

import os
import re
import sys
import math
import json
import random
import zipfile
import shutil
from typing import Dict, List, Tuple, Any, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    PreTrainedTokenizerFast,
    LlamaConfig,
    LlamaForCausalLM,
    MistralConfig,
    MistralForCausalLM,
    get_cosine_schedule_with_warmup,
    get_linear_schedule_with_warmup,
)
from huggingface_hub import HfApi, hf_hub_download, login


## 1. Environment & Hardware Acceleration Setup


In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        login(token=HF_TOKEN)
        print("Logged into Hugging Face Hub successfully.")
    except Exception as e:
        print(f"HF Login notice: {e}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## 2. Canonical L-NER Constants & Hyperparameters


In [ ]:
SUB_ID = "sub04"
SUB_NAME = "Sub 04: Model 1 + Partial Layer-Wise Fine-Tuning"
BASE_MODEL_NAME = "model1-deep-llama-135m"
METHOD = "partial_ft"
LOSS_TYPE = "ce"
DESCRIPTION = "Partial fine-tuning: layers 1 to 8 frozen; top layers 9 to 16 and classification head unfrozen."

ENTITY_LABELS = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]
ALL_LABELS = ENTITY_LABELS + ["O"]
NUM_LABELS = len(ALL_LABELS)  # 13
LABEL2ID = {lbl: i for i, lbl in enumerate(ALL_LABELS)}
ID2LABEL = {i: lbl for i, lbl in enumerate(ALL_LABELS)}
O_INDEX = LABEL2ID["O"]
IGNORE_INDEX = -100

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MASTER_REPO = "sarweshbarnwal/CS781-Legal-Models"
HF_TOKENIZER_REPO = "sarweshbarnwal/legal-tokenizer-32000"

MAX_SEQ_LEN = 512
STRIDE = 64
MICRO_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
PEAK_LR = 0.0001
MIN_LR = 1e-05
EPOCHS = 4
WARMUP_RATIO = 0.08

OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/kaggle/working/cs781_phase2")
EXP_OUTPUT_DIR = os.path.join(OUTPUT_DIR, SUB_ID)
PREDICTION_PATH = os.path.join(EXP_OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(EXP_OUTPUT_DIR, "submission.zip")
os.makedirs(EXP_OUTPUT_DIR, exist_ok=True)

print(f"Initialized Experiment {SUB_ID}: {SUB_NAME}")


## 3. Download Phase 2 Dataset Files & Setup Clean Validation Split


In [ ]:
def download_dataset_files(target_dir: str = "./data"):
    os.makedirs(target_dir, exist_ok=True)
    required_files = [
        ("Phase_2_LNER/Train/labels.txt", "labels.txt"),
        ("Phase_2_LNER/Train/fold1-text.json", "fold1-text.json"),
        ("Phase_2_LNER/Train/fold1-gold.json", "fold1-gold.json"),
        ("Phase_2_LNER/Train/fold2-text.json", "fold2-text.json"),
        ("Phase_2_LNER/Train/fold2-gold.json", "fold2-gold.json"),
        ("Phase_2_LNER/Test/fold3-text.json", "fold3-text.json"),
    ]
    paths = {}
    for hf_subpath, local_filename in required_files:
        local_path = os.path.join(target_dir, local_filename)
        paths[local_filename] = local_path
        if not os.path.exists(local_path):
            try:
                print(f"Downloading {hf_subpath} from {HF_DATASET_REPO}...")
                downloaded = hf_hub_download(
                    repo_id=HF_DATASET_REPO,
                    filename=hf_subpath,
                    repo_type="dataset",
                    token=HF_TOKEN
                )
                shutil.copy(downloaded, local_path)
            except Exception as e:
                print(f"Notice: {hf_subpath} download notice ({e}). Checking local fallback...")
    return paths

DATA_PATHS = download_dataset_files()

def load_json(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

fold1_text = load_json(DATA_PATHS.get("fold1-text.json", ""))
fold1_gold = load_json(DATA_PATHS.get("fold1-gold.json", ""))
fold2_text = load_json(DATA_PATHS.get("fold2-text.json", ""))
fold2_gold = load_json(DATA_PATHS.get("fold2-gold.json", ""))
fold3_text = load_json(DATA_PATHS.get("fold3-text.json", ""))

# Clean train vs validation split: Train on fold1, Validate on fold2 (Zero Data Leakage!)
train_text = fold1_text if fold1_text else {**fold1_text, **fold2_text}
train_gold = fold1_gold if fold1_gold else {**fold1_gold, **fold2_gold}
val_text = fold2_text
val_gold = fold2_gold

print(f"Dataset split: {len(train_text)} train documents (fold1) | {len(val_text)} validation documents (fold2 - zero leakage) | {len(fold3_text)} test documents.")


## 4. Tokenizer & Sliding Window Dataset Pipeline (Custom Collate Fix)


In [ ]:
# Load exact tokenizer from the Phase 1 model subfolder
tokenizer = None
try:
    print(f"Loading custom legal tokenizer from {HF_MASTER_REPO} subfolder: {BASE_MODEL_NAME}/tokenizer ...")
    tokenizer = AutoTokenizer.from_pretrained(
        HF_MASTER_REPO,
        subfolder=f"{BASE_MODEL_NAME}/tokenizer",
        token=HF_TOKEN
    )
    print("Loaded model-specific custom legal tokenizer successfully.")
except Exception as e:
    print(f"Subfolder tokenizer notice: {e}. Trying fallback repos...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_REPO, token=HF_TOKEN)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained("gpt2")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def align_tokens_with_spans(text, spans, offsets):
    labels = []
    for (start_c, end_c) in offsets:
        if start_c == end_c:
            labels.append(IGNORE_INDEX)
            continue
        assigned = O_INDEX
        for span in spans:
            g_start, g_end, g_label = span[0], span[1], span[2]
            if max(start_c, g_start) < min(end_c, g_end):
                if g_label in LABEL2ID:
                    assigned = LABEL2ID[g_label]
                    break
        labels.append(assigned)
    return labels

class SlidingWindowNERDataset(Dataset):
    def __init__(self, docs, golds=None, tokenizer=tokenizer, max_len=MAX_SEQ_LEN, stride=STRIDE):
        self.samples = []
        for doc_id, text in docs.items():
            doc_golds = golds.get(doc_id, []) if golds is not None else None
            encoding = tokenizer(
                text,
                truncation=True,
                max_length=max_len,
                stride=stride,
                return_overflowing_tokens=True,
                return_offsets_mapping=True,
                padding="max_length",
                return_tensors="pt"
            )
            num_windows = encoding["input_ids"].shape[0]
            for w in range(num_windows):
                input_ids = encoding["input_ids"][w]
                attention_mask = encoding["attention_mask"][w]
                offsets = encoding["offset_mapping"][w].tolist()

                if doc_golds is not None:
                    labels = align_tokens_with_spans(text, doc_golds, offsets)
                    labels_tensor = torch.tensor(labels, dtype=torch.long)
                else:
                    labels_tensor = torch.full_like(input_ids, IGNORE_INDEX)

                self.samples.append({
                    "doc_id": doc_id,
                    "window_idx": w,
                    "input_ids": input_ids,
                    "attention_mask": attention_mask,
                    "labels": labels_tensor,
                    "offsets": offsets,
                    "text_len": len(text)
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def ner_collate_fn(batch):
    """Custom collation: stacks tensors, preserves offsets and doc_id as raw lists."""
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
        "doc_id": [b["doc_id"] for b in batch],
        "offsets": [b["offsets"] for b in batch],
        "text_len": [b["text_len"] for b in batch]
    }

train_dataset = SlidingWindowNERDataset(train_text, train_gold)
val_dataset = SlidingWindowNERDataset(val_text, val_gold)
print(f"Generated {len(train_dataset)} training sliding-window samples, {len(val_dataset)} validation samples.")

# Compute Inverse Class Frequency weights
class_counts = torch.zeros(NUM_LABELS, dtype=torch.float32)
for s in train_dataset.samples:
    valid_lbls = s["labels"][s["labels"] != IGNORE_INDEX]
    for c in range(NUM_LABELS):
        class_counts[c] += (valid_lbls == c).sum().item()
total_tokens = class_counts.sum().item()
class_weights = torch.zeros(NUM_LABELS, dtype=torch.float32)
for c in range(NUM_LABELS):
    if class_counts[c] > 0:
        class_weights[c] = (total_tokens / (NUM_LABELS * class_counts[c])) ** 0.5
    else:
        class_weights[c] = 1.0
class_weights = (class_weights / class_weights.mean()).to(device)


## 5. Model Architecture & Backbone Instantiation (Fix: subfolder= & fp32 base)


In [ ]:
def load_or_create_backbone():
    print(f"Loading Phase 1 pretrained backbone from {HF_MASTER_REPO} subfolder: {BASE_MODEL_NAME} ...")
    try:
        backbone = AutoModelForCausalLM.from_pretrained(
            HF_MASTER_REPO,
            subfolder=BASE_MODEL_NAME,
            token=HF_TOKEN,
            torch_dtype=torch.float32,  # Base model in fp32 for stable AMP and GradScaler
            trust_remote_code=True
        )
        print(f"Successfully loaded Phase 1 pretrained backbone from subfolder {BASE_MODEL_NAME}!")
        return backbone
    except Exception as e:
        print(f"Hub loading notice: {e}. Initializing architecture fallback...")
        cfg = LlamaConfig(
            vocab_size=len(tokenizer),
            hidden_size=768,
            num_hidden_layers=16,
            num_attention_heads=12,
            num_key_value_heads=4,
            intermediate_size=2304,
            max_position_embeddings=MAX_SEQ_LEN * 2,
            rms_norm_eps=1e-6,
            tie_word_embeddings=True
        )
        return LlamaForCausalLM(cfg)

class LNERDecoderClassifier(nn.Module):
    def __init__(self, backbone, hidden_size=768, num_labels=NUM_LABELS, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.hidden_size = hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids, attention_mask=None, labels=None):
        if hasattr(self.backbone, "model") and hasattr(self.backbone.model, "layers"):
            outputs = self.backbone.model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
            h = outputs.last_hidden_state
        else:
            outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True, return_dict=True)
            h = outputs.hidden_states[-1] if hasattr(outputs, "hidden_states") else outputs[0]
        h = self.dropout(h)
        logits = self.classifier(h)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.view(-1, NUM_LABELS), labels.view(-1), ignore_index=IGNORE_INDEX)
        return logits, loss


## 6. Adaptation Injection: True Multiplicative LoRMA vs Additive LoRA


In [ ]:
class LoRALinear(nn.Module):
    """Standard Additive LoRA (Hu et al.): y = W_0 x + (alpha/r) * B(A x)"""
    def __init__(self, original_linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.scaling = alpha / r
        self.weight = original_linear.weight
        self.weight.requires_grad = False
        self.bias = original_linear.bias
        if self.bias is not None:
            self.bias.requires_grad = False
        self.lora_A = nn.Parameter(torch.zeros(r, self.in_features))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, r))
        self.lora_dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        base_out = F.linear(x, self.weight, self.bias)
        lora_out = (self.lora_dropout(x) @ self.lora_A.t()) @ self.lora_B.t()
        return base_out + self.scaling * lora_out


class LoRMALinear(nn.Module):
    """
    Exploration Lab Multiplicative Low-Rank Matrix Adaptation (LoRMA):
    y = (I + (alpha/r) * BA) (W_0 x)
    h_base = W_0 x;  y = h_base + (alpha/r) * B(A(h_base))
    A: (r, d_out), B: (d_out, r) modulating the pre-activation base output!
    Reference: github.com/Exploration-Lab/LoRMA (Prof. Ashutosh Modi, IIT Kanpur)
    """
    def __init__(self, original_linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.scaling = alpha / r
        self.weight = original_linear.weight
        self.weight.requires_grad = False
        self.bias = original_linear.bias
        if self.bias is not None:
            self.bias.requires_grad = False
        self.lorma_A = nn.Parameter(torch.zeros(r, self.out_features))
        self.lorma_B = nn.Parameter(torch.zeros(self.out_features, r))
        self.lorma_dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B)

    def forward(self, x):
        h_base = F.linear(x, self.weight, self.bias)
        h_mod = (self.lorma_dropout(h_base) @ self.lorma_A.t()) @ self.lorma_B.t()
        return h_base + self.scaling * h_mod


class ResidualBottleneckAdapter(nn.Module):
    def __init__(self, d_model=768, reduction=16):
        super().__init__()
        bottleneck_dim = max(16, d_model // reduction)
        self.down = nn.Linear(d_model, bottleneck_dim)
        self.act = nn.SiLU()
        self.up = nn.Linear(bottleneck_dim, d_model)
        nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(self.act(self.down(x)))


def configure_model_adaptation(model):
    method = "partial_ft"
    target_modules = []

    if method == "lorma":
        for param in model.parameters():
            param.requires_grad = False
        replaced = 0
        for name, module in list(model.named_modules()):
            for target in target_modules:
                if target in name and isinstance(module, nn.Linear) and not isinstance(module, (LoRALinear, LoRMALinear)):
                    parts = name.split(".")
                    parent = model
                    for part in parts[:-1]:
                        parent = getattr(parent, part)
                    child = parts[-1]
                    setattr(parent, child, LoRMALinear(module))
                    replaced += 1
                    break
        for param in model.classifier.parameters():
            param.requires_grad = True
        print(f"Applied Multiplicative LoRMA: Replaced {replaced} linear layers with output-modulation factors.")

    elif method == "lora":
        for param in model.parameters():
            param.requires_grad = False
        replaced = 0
        for name, module in list(model.named_modules()):
            for target in target_modules:
                if target in name and isinstance(module, nn.Linear) and not isinstance(module, (LoRALinear, LoRMALinear)):
                    parts = name.split(".")
                    parent = model
                    for part in parts[:-1]:
                        parent = getattr(parent, part)
                    child = parts[-1]
                    setattr(parent, child, LoRALinear(module))
                    replaced += 1
                    break
        for param in model.classifier.parameters():
            param.requires_grad = True
        print(f"Applied Additive LoRA: Replaced {replaced} linear layers with low-rank input-adaptation factors.")

    elif method == "bitfit":
        for name, param in model.named_parameters():
            if "classifier" in name or param.ndim == 1 or "bias" in name:
                param.requires_grad = True
            else:
                param.requires_grad = False
        print("Applied BitFit: Only 1D bias parameters and classification head unfrozen.")

    elif method == "partial_ft":
        for param in model.parameters():
            param.requires_grad = False
        layers = model.backbone.model.layers if hasattr(model.backbone, "model") else model.backbone.layers
        for i, l in enumerate(layers):
            if i >= 8:
                for p in l.parameters():
                    p.requires_grad = True
        for param in model.classifier.parameters():
            param.requires_grad = True
        print("Applied Partial FT: Layers 1..8 frozen; top layers unfrozen.")

    elif method == "full_ft":
        for param in model.parameters():
            param.requires_grad = True
        print("Applied Full FT: All parameters unfrozen.")

    elif method == "bottleneck_adapters":
        for param in model.parameters():
            param.requires_grad = False
        layers = model.backbone.model.layers if hasattr(model.backbone, "model") else model.backbone.layers
        for l in layers:
            l.adapter = ResidualBottleneckAdapter()
            for p in l.adapter.parameters():
                p.requires_grad = True
        for param in model.classifier.parameters():
            param.requires_grad = True
        print("Applied Bottleneck Adapters: Injected into transformer blocks with frozen backbone.")

    elif method in ["distillation", "ensemble_distill"]:
        for param in model.parameters():
            param.requires_grad = True
        print("Configured Student Model for Knowledge Distillation.")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Total Parameters: {total:,} | Trainable Parameters: {trainable:,} ({100*trainable/total:.2f}%)")
    return trainable

raw_backbone = load_or_create_backbone()
model = LNERDecoderClassifier(raw_backbone).to(device)
configure_model_adaptation(model)

# Instantiate Teacher Model for Knowledge Distillation
teacher_model = None
teacher_model2 = None
if METHOD == "distillation":
    print(f"Loading Teacher Model (model1-deep-llama-135m)...")
    try:
        t_bb = AutoModelForCausalLM.from_pretrained(
            HF_MASTER_REPO,
            subfolder="model1-deep-llama-135m",
            token=HF_TOKEN,
            torch_dtype=torch.float32,
            trust_remote_code=True
        )
        teacher_model = LNERDecoderClassifier(t_bb, hidden_size=768).to(device)
        teacher_model.eval()
        for p in teacher_model.parameters():
            p.requires_grad = False
        print(f"Successfully loaded and initialized Teacher Model (model1-deep-llama-135m)!")
    except Exception as e:
        print(f"Notice: Teacher model download notice ({e}). Using student copy as reference teacher.")
        teacher_model = model

elif METHOD == "ensemble_distill":
    print(f"Loading Ensemble Teacher Models (model1-deep-llama-135m & model4-capacity-boundary-154m)...")
    try:
        t1_bb = AutoModelForCausalLM.from_pretrained(HF_MASTER_REPO, subfolder="model1-deep-llama-135m", token=HF_TOKEN, torch_dtype=torch.float32, trust_remote_code=True)
        teacher_model = LNERDecoderClassifier(t1_bb, hidden_size=768).to(device)
        teacher_model.eval()
        for p in teacher_model.parameters(): p.requires_grad = False

        t2_bb = AutoModelForCausalLM.from_pretrained(HF_MASTER_REPO, subfolder="model4-capacity-boundary-154m", token=HF_TOKEN, torch_dtype=torch.float32, trust_remote_code=True)
        teacher_model2 = LNERDecoderClassifier(t2_bb, hidden_size=896).to(device)
        teacher_model2.eval()
        for p in teacher_model2.parameters(): p.requires_grad = False
        print("Successfully loaded both ensemble teacher models!")
    except Exception as e:
        print(f"Notice: Ensemble teacher loading notice ({e}).")
        teacher_model = model


## 7. Loss Function Setup


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        logits = logits.view(-1, NUM_LABELS)
        targets = targets.view(-1)
        valid = targets != IGNORE_INDEX
        if not valid.any():
            return torch.tensor(0.0, device=logits.device, requires_grad=True)
        ce = F.cross_entropy(logits[valid], targets[valid], reduction="none")
        p_t = torch.exp(-ce)
        focal = self.alpha * ((1.0 - p_t) ** self.gamma) * ce
        return focal.mean()

class DistillLossModule(nn.Module):
    def __init__(self, temp=2.0, a_ce=0.4, a_kd=0.6):
        super().__init__()
        self.temp = temp
        self.a_ce = a_ce
        self.a_kd = a_kd

    def forward(self, s_logits, t_logits, targets):
        loss_ce = F.cross_entropy(s_logits.view(-1, NUM_LABELS), targets.view(-1), ignore_index=IGNORE_INDEX)
        valid = targets.view(-1) != IGNORE_INDEX
        if not valid.any():
            return loss_ce
        s_log_p = F.log_softmax(s_logits.view(-1, NUM_LABELS)[valid] / self.temp, dim=-1)
        t_p = F.softmax(t_logits.view(-1, NUM_LABELS)[valid] / self.temp, dim=-1)
        loss_kd = F.kl_div(s_log_p, t_p, reduction="batchmean") * (self.temp ** 2)
        return self.a_ce * loss_ce + self.a_kd * loss_kd

def compute_loss(logits, targets, teacher_logits=None):
    loss_name = "ce"
    if loss_name == "class_weighted":
        return F.cross_entropy(logits.view(-1, NUM_LABELS), targets.view(-1), weight=class_weights, ignore_index=IGNORE_INDEX)
    elif loss_name == "focal":
        return FocalLoss()(logits, targets)
    elif loss_name == "distill" and teacher_logits is not None:
        return DistillLossModule()(logits, teacher_logits, targets)
    else:
        return F.cross_entropy(logits.view(-1, NUM_LABELS), targets.view(-1), ignore_index=IGNORE_INDEX)


## 8. Training Optimization Loop (AMP with fp32 parameters)


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=True, collate_fn=ner_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=False, collate_fn=ner_collate_fn)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=PEAK_LR,
    betas=(0.9, 0.999),
    weight_decay=0.01
)

total_steps = (len(train_loader) // GRADIENT_ACCUMULATION_STEPS) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

if "cosine" == "linear":
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
else:
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f"Starting training: {EPOCHS} epochs | {total_steps} total steps | Peak LR={PEAK_LR}")

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits, _ = model(input_ids, attention_mask=attention_mask)
            
            # Extract teacher logits if distillation is enabled
            t_logits = None
            if METHOD == "distillation" and teacher_model is not None:
                with torch.no_grad():
                    t_logits, _ = teacher_model(input_ids, attention_mask=attention_mask)
            elif METHOD == "ensemble_distill" and teacher_model is not None:
                with torch.no_grad():
                    t1_logits, _ = teacher_model(input_ids, attention_mask=attention_mask)
                    t2_logits, _ = teacher_model2(input_ids, attention_mask=attention_mask) if teacher_model2 is not None else (t1_logits, None)
                    t_logits = 0.5 * (t1_logits + t2_logits)

            loss = compute_loss(logits, labels, teacher_logits=t_logits)
            loss = loss / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(loss).backward()
        running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

    avg_train_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch}/{EPOCHS} - Train Loss: {avg_train_loss:.4f}")


## 9. Validation Exact Span & Token-Level Macro-F1 Evaluation


In [ ]:
model.eval()
tp_tok = {lbl: 0 for lbl in ENTITY_LABELS}
fp_tok = {lbl: 0 for lbl in ENTITY_LABELS}
fn_tok = {lbl: 0 for lbl in ENTITY_LABELS}

val_doc_windows = {doc_id: [] for doc_id in val_text.keys()}

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        doc_ids = batch["doc_id"]
        offsets_list = batch["offsets"]

        logits, _ = model(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(logits, dim=-1)
        probs = F.softmax(logits, dim=-1).cpu()

        # Token-level stats
        for p_seq, t_seq in zip(preds.cpu().tolist(), labels.cpu().tolist()):
            for p, t in zip(p_seq, t_seq):
                if t == IGNORE_INDEX:
                    continue
                p_lbl = ID2LABEL.get(p, "O")
                t_lbl = ID2LABEL.get(t, "O")
                if t_lbl in tp_tok:
                    if p_lbl == t_lbl:
                        tp_tok[t_lbl] += 1
                    else:
                        fn_tok[t_lbl] += 1
                        if p_lbl in fp_tok:
                            fp_tok[p_lbl] += 1
                else:
                    if p_lbl in fp_tok:
                        fp_tok[p_lbl] += 1

        for d_id, offs, p_vec in zip(doc_ids, offsets_list, probs):
            val_doc_windows[d_id].append({"offsets": offs, "probs": p_vec})

per_class_f1 = {}
tok_f1_list = []
print("\n--- Validation Token-Level Macro-F1 ---")
for lbl in ENTITY_LABELS:
    tot_p = tp_tok[lbl] + fp_tok[lbl]
    tot_g = tp_tok[lbl] + fn_tok[lbl]
    prec = (tp_tok[lbl] / tot_p) if tot_p > 0 else 0.0
    rec = (tp_tok[lbl] / tot_g) if tot_g > 0 else 0.0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
    per_class_f1[lbl] = f1
    tok_f1_list.append(f1)
    print(f"  {lbl:<10} | Prec: {prec:.4f} | Rec: {rec:.4f} | Token-F1: {f1:.4f}")

val_token_macro_f1 = sum(tok_f1_list) / len(tok_f1_list)

# Exact Character-Span Validation Matching against val_gold (fold2)
val_pred_spans = {}
for doc_id, text in val_text.items():
    windows = val_doc_windows.get(doc_id, [])
    text_len = len(text)
    char_probs = torch.zeros((text_len, NUM_LABELS), dtype=torch.float32)
    char_counts = torch.zeros(text_len, dtype=torch.float32)

    for item in windows:
        for (sc, ec), p_vec in zip(item["offsets"], item["probs"]):
            if sc == ec or sc >= text_len:
                continue
            char_probs[sc:ec] += p_vec
            char_counts[sc:ec] += 1.0

    valid_mask = char_counts > 0
    char_probs[valid_mask] = char_probs[valid_mask] / char_counts[valid_mask].unsqueeze(-1)
    char_labels = torch.argmax(char_probs, dim=-1).tolist()

    spans = []
    curr_lbl, curr_st = None, None
    for i, lbl_id in enumerate(char_labels):
        lbl_str = ID2LABEL[lbl_id]
        if lbl_str != "O" and valid_mask[i]:
            if curr_lbl is None:
                curr_st, curr_lbl = i, lbl_str
            elif curr_lbl != lbl_str:
                spans.append([curr_st, i, curr_lbl])
                curr_st, curr_lbl = i, lbl_str
        else:
            if curr_lbl is not None:
                spans.append([curr_st, i, curr_lbl])
                curr_lbl, curr_st = None, None
    if curr_lbl is not None:
        spans.append([curr_st, text_len, curr_lbl])

    cleaned_spans = []
    for s, e, lbl in spans:
        span_txt = text[s:e]
        clean_txt = span_txt.strip()
        if clean_txt:
            lead = len(span_txt) - len(span_txt.lstrip())
            trail = len(span_txt) - len(span_txt.rstrip())
            cs, ce = s + lead, e - trail
            if ce > cs:
                cleaned_spans.append([cs, ce, lbl])
    val_pred_spans[doc_id] = cleaned_spans

# Exact span matching metrics
span_tp = {lbl: 0 for lbl in ENTITY_LABELS}
span_fp = {lbl: 0 for lbl in ENTITY_LABELS}
span_fn = {lbl: 0 for lbl in ENTITY_LABELS}

for doc_id, gold_list in val_gold.items():
    gold_set = set((s[0], s[1], s[2]) for s in gold_list if len(s) >= 3 and s[2] in ENTITY_LABELS)
    pred_set = set((s[0], s[1], s[2]) for s in val_pred_spans.get(doc_id, []) if len(s) >= 3 and s[2] in ENTITY_LABELS)

    for sp in pred_set:
        if sp in gold_set:
            span_tp[sp[2]] += 1
        else:
            span_fp[sp[2]] += 1
    for sp in gold_set:
        if sp not in pred_set:
            span_fn[sp[2]] += 1

span_f1_list = []
print("\n--- Validation Exact Character-Span Macro-F1 (CodaBench Metric) ---")
for lbl in ENTITY_LABELS:
    tot_p = span_tp[lbl] + span_fp[lbl]
    tot_g = span_tp[lbl] + span_fn[lbl]
    prec = (span_tp[lbl] / tot_p) if tot_p > 0 else 0.0
    rec = (span_tp[lbl] / tot_g) if tot_g > 0 else 0.0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
    span_f1_list.append(f1)
    print(f"  {lbl:<10} | Prec: {prec:.4f} | Rec: {rec:.4f} | Span-F1: {f1:.4f}")

val_macro_f1 = sum(span_f1_list) / len(span_f1_list)
print(f"\n🏆 Overall Validation Exact Span Strict Macro-F1: {val_macro_f1:.4f}")


## 10. Test Inference & Span Reconstruction (fold3-text.json)


In [ ]:
print(f"Generating test predictions on {len(fold3_text)} documents from fold3-text.json...")
prediction_dict = {}

test_dataset = SlidingWindowNERDataset(fold3_text, None)
test_loader = DataLoader(test_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=False, collate_fn=ner_collate_fn)

doc_window_preds = {doc_id: [] for doc_id in fold3_text.keys()}

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        doc_ids = batch["doc_id"]
        offsets_list = batch["offsets"]

        logits, _ = model(input_ids, attention_mask=attention_mask)
        probs = F.softmax(logits, dim=-1).cpu()

        for d_id, offs, p in zip(doc_ids, offsets_list, probs):
            doc_window_preds[d_id].append({
                "offsets": offs,
                "probs": p
            })

# Reconstruct document character spans cleanly
for doc_id, text in fold3_text.items():
    windows = doc_window_preds.get(doc_id, [])
    text_len = len(text)
    char_probs = torch.zeros((text_len, NUM_LABELS), dtype=torch.float32)
    char_counts = torch.zeros(text_len, dtype=torch.float32)

    for item in windows:
        for (sc, ec), p_vec in zip(item["offsets"], item["probs"]):
            if sc == ec or sc >= text_len:
                continue
            char_probs[sc:ec] += p_vec
            char_counts[sc:ec] += 1.0

    valid_mask = char_counts > 0
    char_probs[valid_mask] = char_probs[valid_mask] / char_counts[valid_mask].unsqueeze(-1)
    char_labels = torch.argmax(char_probs, dim=-1).tolist()

    spans = []
    curr_lbl, curr_st = None, None
    for i, lbl_id in enumerate(char_labels):
        lbl_str = ID2LABEL[lbl_id]
        if lbl_str != "O" and valid_mask[i]:
            if curr_lbl is None:
                curr_st, curr_lbl = i, lbl_str
            elif curr_lbl != lbl_str:
                spans.append([curr_st, i, curr_lbl])
                curr_st, curr_lbl = i, lbl_str
        else:
            if curr_lbl is not None:
                spans.append([curr_st, i, curr_lbl])
                curr_lbl, curr_st = None, None
    if curr_lbl is not None:
        spans.append([curr_st, text_len, curr_lbl])

    cleaned_spans = []
    for s, e, lbl in spans:
        span_txt = text[s:e]
        clean_txt = span_txt.strip()
        if clean_txt:
            lead = len(span_txt) - len(span_txt.lstrip())
            trail = len(span_txt) - len(span_txt.rstrip())
            cs, ce = s + lead, e - trail
            if ce > cs:
                cleaned_spans.append([cs, ce, lbl])

    prediction_dict[doc_id] = cleaned_spans

# Save prediction.json
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(prediction_dict, f, indent=2)
print(f"Saved prediction.json to: {PREDICTION_PATH}")

# Pack submission.zip
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")
print(f"Generated submission.zip at: {SUBMISSION_ZIP}")


## 11. Save Checkpoints & Sync All Artifacts to Hugging Face Hub (Phase_2/...)


In [ ]:
SUB_FOLDER = f"{SUB_ID}_{BASE_MODEL_NAME.replace('-', '_')}_{METHOD}"
HF_PHASE2_REPO_PATH = f"Phase_2/{SUB_FOLDER}"

# 1. Save trained model weights & head
WEIGHTS_PATH = os.path.join(EXP_OUTPUT_DIR, "model_weights.pt")
torch.save({
    "sub_id": SUB_ID,
    "method": METHOD,
    "base_model": BASE_MODEL_NAME,
    "state_dict": model.state_dict(),
    "classifier_state_dict": model.classifier.state_dict() if hasattr(model, "classifier") else None,
    "val_macro_f1": val_macro_f1,
    "val_token_macro_f1": val_token_macro_f1
}, WEIGHTS_PATH)
print(f"Saved trained weights to: {WEIGHTS_PATH}")

# 2. Save validation metrics
METRICS_PATH = os.path.join(EXP_OUTPUT_DIR, "val_metrics.json")
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "sub_id": SUB_ID,
        "sub_name": SUB_NAME,
        "base_model": BASE_MODEL_NAME,
        "method": METHOD,
        "val_exact_span_macro_f1": val_macro_f1,
        "val_token_macro_f1": val_token_macro_f1,
        "per_class_f1": per_class_f1,
        "epochs": EPOCHS,
        "peak_lr": PEAK_LR
    }, f, indent=2)
print(f"Saved validation metrics to: {METRICS_PATH}")

# 3. Save experiment configuration
CONFIG_PATH = os.path.join(EXP_OUTPUT_DIR, "config.json")
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "sub_id": SUB_ID,
        "sub_name": SUB_NAME,
        "base_model": BASE_MODEL_NAME,
        "method": METHOD,
        "loss_type": LOSS_TYPE,
        "max_seq_len": MAX_SEQ_LEN,
        "stride": STRIDE,
        "micro_batch_size": MICRO_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch_size": MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "peak_lr": PEAK_LR,
        "min_lr": MIN_LR,
        "epochs": EPOCHS,
        "warmup_ratio": WARMUP_RATIO,
        "total_parameters": sum(p.numel() for p in model.parameters()),
        "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad)
    }, f, indent=2)
print(f"Saved configuration to: {CONFIG_PATH}")

# 4. Create README.md for this submission folder
SUB_README_PATH = os.path.join(EXP_OUTPUT_DIR, "README.md")
with open(SUB_README_PATH, "w", encoding="utf-8") as f:
    f.write(f"""# {SUB_NAME}
- **Base Model Backbone:** `{BASE_MODEL_NAME}`
- **Adaptation Technique:** `{METHOD.upper()}`
- **Validation Exact Span Strict Macro-F1:** `{val_macro_f1:.4f}`
- **Validation Token Macro-F1:** `{val_token_macro_f1:.4f}`
- **Target Objective:** Strict Macro-F1 Leaderboard Champion

### Files in this Submission Folder:
- `prediction.json`: CodaBench test predictions on `fold3-text.json`
- `submission.zip`: Packaged zip archive ready for CodaBench leaderboard submission
- `val_metrics.json`: Detailed validation Strict Macro-F1 and per-class metrics
- `config.json`: Hyperparameter, architectural, and optimization configuration
- `model_weights.pt`: Saved weights/adapters for downstream evaluation or distillation
""")

# 5. Push entire submission directory to Hugging Face Hub under Phase_2/{SUB_FOLDER}
print(f"\nPushing all Phase 2 artifacts to Hugging Face Hub: {HF_MASTER_REPO}/{HF_PHASE2_REPO_PATH} ...")
try:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HF_MASTER_REPO, repo_type="model", exist_ok=True)
    api.upload_folder(
        folder_path=EXP_OUTPUT_DIR,
        path_in_repo=HF_PHASE2_REPO_PATH,
        repo_id=HF_MASTER_REPO,
        commit_message=f"Upload Phase 2 artifacts for {SUB_ID} ({METHOD}) - Macro-F1: {val_macro_f1:.4f}"
    )
    print(f"Successfully pushed all artifacts to Hugging Face Hub!")
    print(f"View files online: https://huggingface.co/{HF_MASTER_REPO}/tree/main/{HF_PHASE2_REPO_PATH}")
except Exception as e:
    print(f"Notice: Hugging Face upload notice: {e}")
    print("Ensure your HF write token is set in Kaggle Secrets with label 'HF'.")

print(f"Submission {SUB_ID} completed successfully!")
